# Experiment 1: Global Routing Dynamic Quantization

This experiment implements a single, global decision (INT5 vs INT7) per image to dramatically reduce inference latency compared to the original per-layer DQNet.

During inference, the batch is dynamically split into an "Easy" (INT5) batch and a "Hard" (INT7) batch, meaning only **one convolution** is ever executed per image per layer!

In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
import sys
sys.path.append('Quantization_project')

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time, numpy as np, matplotlib.pyplot as plt

# Import our new Global Routing architecture
from experiment1 import global_dq_resnet20, compute_global_bitops, max_global_bitops, BIT_OPTIONS

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

## Training Setup

In [ ]:
model = global_dq_resnet20().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

LAMBDA = 0.0001
EPOCHS = 200
TEMP_START = 1.0
TEMP_END = 0.1
fp32_max = max_global_bitops(model.layer_flops)

> **Note:** To see the latency improvements immediately, you can interrupt the training loop after a few epochs. We have added a latency evaluation cell below.

In [ ]:
train_losses, train_accs, test_accs, avg_bitops_log = [], [], [], []

try:
    for epoch in range(EPOCHS):
        model.train()
        temp = TEMP_START - (TEMP_START - TEMP_END) * epoch / max(EPOCHS - 1, 1)
        running_loss, correct, total, epoch_bitops = 0, 0, 0, 0

        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            # Forward pass (soft routing)
            outputs, soft_bits = model(inputs, temperature=temp)
            ce_loss = criterion(outputs, targets)
            
            # BitOPs penalty
            bitops = compute_global_bitops(soft_bits, model.layer_flops)
            bitops_penalty = bitops / fp32_max
            loss = ce_loss + LAMBDA * bitops_penalty
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            epoch_bitops += bitops.item()

        scheduler.step()
        train_acc = 100. * correct / total
        train_losses.append(running_loss / len(trainloader))
        train_accs.append(train_acc)

        # Evaluation
        model.eval()
        t_correct, t_total = 0, 0
        int_easy_count, int_hard_count = 0, 0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs, hard_idx = model(inputs) # Hard inference routing
                
                _, predicted = outputs.max(1)
                t_total += targets.size(0)
                t_correct += predicted.eq(targets).sum().item()
                
                int_easy_count += (hard_idx == 0).sum().item()
                int_hard_count += (hard_idx == 1).sum().item()
                
        test_acc = 100. * t_correct / t_total
        test_accs.append(test_acc)
        avg_bo = epoch_bitops / len(trainloader) / fp32_max
        avg_bitops_log.append(avg_bo)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_losses[-1]:.3f} | Train: {train_acc:.2f}% | Test: {test_acc:.2f}% | Temp: {temp:.3f}')
            print(f'  -> Routing Stats (Test): INT{BIT_OPTIONS[0]} (Easy): {int_easy_count} ({int_easy_count/t_total*100:.1f}%), INT{BIT_OPTIONS[1]} (Hard): {int_hard_count} ({int_hard_count/t_total*100:.1f}%)')
except KeyboardInterrupt:
    print("\nTraining interrupted early!")
    
torch.save(model.state_dict(), 'global_dq_resnet20.pth')
print('Model saved.')

## Latency Benchmark

In [ ]:
import time
model.eval()

times = []
with torch.no_grad():
    # Warmup
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _ = model(inputs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for _ in range(30):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        for inputs, _ in testloader:
            inputs = inputs.to(device)
            _ = model(inputs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.time() - start)

print(f'Global DQNet Avg Inference Time (30 runs): {np.mean(times):.4f} ± {np.std(times):.4f}s')